<a href="https://colab.research.google.com/github/nurkaussar/assignment-06-rnn-foundations-pytorch/blob/main/assignment6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Part B: Reproduce the Same RNN in PyTorch

In [1]:
import torch
import torch.nn as nn

# Input sequences from Part A
# x1=[1,0,0], x2=[0,1,0], x3=[0,0,1], x4=[1,1,0]
x = torch.tensor([[[1.0, 0.0, 0.0],
                   [0.0, 1.0, 0.0],
                   [0.0, 0.0, 1.0],
                   [1.0, 1.0, 0.0]]])

print(f"Shape of x: {x.shape}") # Expected: (1, 4, 3)

Shape of x: torch.Size([1, 4, 3])


In [2]:
# Initialize RNN: input_size=3, hidden_size=2
rnn = nn.RNN(input_size=3, hidden_size=2, batch_first=True)

# Manually set weights and biases to match Part A
with torch.no_grad():
    # Input-to-hidden weights (Wx)
    rnn.weight_ih_l0 = nn.Parameter(torch.tensor([[0.5, -0.2, 0.1],
                                                 [0.0, 0.3, -0.4]]))
    # Hidden-to-hidden weights (Wh)
    rnn.weight_hh_l0 = nn.Parameter(torch.tensor([[0.2, 0.1],
                                                 [-0.3, 0.4]]))
    # Biases (b)
    # PyTorch has two bias terms (ih and hh), we put the values in ih and keep hh as zero
    rnn.bias_ih_l0 = nn.Parameter(torch.tensor([0.0, 0.1]))
    rnn.bias_hh_l0 = nn.Parameter(torch.tensor([0.0, 0.0]))

In [3]:
# Perform the forward pass
output, h_n = rnn(x)

print("--- Tensor Shapes ---")
print(f"Output shape: {output.shape}") # (batch, seq_len, hidden_size)
print(f"h_n shape: {h_n.shape}")      # (num_layers, batch, hidden_size)

print("\n--- Numerical Results ---")
print("Hidden states for all steps (output):")
print(output)

print("\nFinal hidden state (h_n):")
print(h_n)

--- Tensor Shapes ---
Output shape: torch.Size([1, 4, 2])
h_n shape: torch.Size([1, 1, 2])

--- Numerical Results ---
Hidden states for all steps (output):
tensor([[[ 0.4621,  0.0997],
         [-0.0973,  0.2924],
         [ 0.1093, -0.1526],
         [ 0.2973,  0.2969]]], grad_fn=<TransposeBackward1>)

Final hidden state (h_n):
tensor([[[0.2973, 0.2969]]], grad_fn=<StackBackward0>)


**output[:, -1, :]:** Represents the hidden state of the very last time step in the sequence for all batches.

**h_n[-1]:** Represents the final hidden state of the last RNN layer.

**Why are they the same?:** In a single-layer RNN, the output at the final time step is identical to the final hidden state passed to the next potential layer.

**batch_first=False:** If set to False, PyTorch expects the input shape to be (seq_len, batch_size, input_size).

#Part C: DNA Sequence Classifier

###C1. Generate Data

In [4]:
import random

def generate_dna_sequence(min_len, max_len, motif="ATG", positive=True):
    """
    Generates a random DNA sequence.
    If positive=True, ensures the motif is present.
    If positive=False, ensures the motif is absent.
    """
    length = random.randint(min_len, max_len)
    bases = ["A", "C", "G", "T"]

    while True:
        # Create a random sequence
        seq = "".join(random.choice(bases) for _ in range(length))

        if positive:
            # Insert motif at a random position
            start_idx = random.randint(0, length - len(motif))
            seq_list = list(seq)
            seq_list[start_idx : start_idx + len(motif)] = list(motif)
            seq = "".join(seq_list)
            return seq
        else:
            # If the random sequence accidentally contains the motif, try again
            if motif not in seq:
                return seq

def create_dataset(num_samples, motif="ATG"):
    """
    Creates a balanced dataset with 50% positive and 50% negative samples.
    """
    data = []
    # 50/50 balance is usually best for training
    for _ in range(num_samples // 2):
        data.append((generate_dna_sequence(30, 80, motif, True), 1))
        data.append((generate_dna_sequence(30, 80, motif, False), 0))

    random.shuffle(data)
    return data

In [5]:
# Set seed for reproducibility
random.seed(42)

# Generate data
train_data = create_dataset(2000)
val_data = create_dataset(500)
test_data = create_dataset(500)

# Preview the data
print(f"Total training samples: {len(train_data)}")
print(f"Sample sequence: {train_data[0][0]}")
print(f"Sample label: {train_data[0][1]}")

# Quick check on labels balance
train_labels = [label for seq, label in train_data]
print(f"Positive samples in train: {sum(train_labels)}")

Total training samples: 2000
Sample sequence: AACATCGGAGAATTGAGAGTGTAACGCAGCCCAGCCCCCGTCGTC
Sample label: 0
Positive samples in train: 1000


###C2. Encode and Pad

In [6]:
import torch
from torch.nn.utils.rnn import pad_sequence

# Mapping based on assignment instructions
char2idx = {"A": 0, "C": 1, "G": 2, "T": 3, "<PAD>": 4}

def encode_sequence(seq, mapping):
    """Converts a string of DNA to a list of integers."""
    return torch.tensor([mapping[char] for char in seq])

def prepare_tensors(data, mapping):
    """
    Encodes, pads sequences, and creates a tensor for lengths.
    Required for nn.utils.rnn.pack_padded_sequence later.
    """
    sequences = [encode_sequence(seq, mapping) for seq, label in data]
    labels = torch.tensor([label for seq, label in data], dtype=torch.long)

    # Store the actual lengths before padding
    lengths = torch.tensor([len(s) for s in sequences])

    # Pad sequences with the value 4 (<PAD>)
    # batch_first=True makes the shape (batch_size, max_seq_len)
    padded_sequences = pad_sequence(sequences, batch_first=True, padding_value=mapping["<PAD>"])

    return padded_sequences, labels, lengths

# Process our datasets
X_train, y_train, len_train = prepare_tensors(train_data, char2idx)
X_val, y_val, len_val = prepare_tensors(val_data, char2idx)
X_test, y_test, len_test = prepare_tensors(test_data, char2idx)

print("--- Shapes ---")
print(f"X_train shape: {X_train.shape}") # Should be (2000, max_length_in_batch)
print(f"Lengths shape: {len_train.shape}")
print(f"y_train shape: {y_train.shape}")

print("\nExample of padded sequence (first 10 elements):")
print(X_train[0][:10])

--- Shapes ---
X_train shape: torch.Size([2000, 80])
Lengths shape: torch.Size([2000])
y_train shape: torch.Size([2000])

Example of padded sequence (first 10 elements):
tensor([0, 0, 1, 0, 3, 1, 2, 2, 0, 2])


###C3. Train a Classifier

In [7]:
class DNAClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes, pad_idx):
        super().__init__()
        # Converts word indices (0-4) into dense vectors
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # The RNN layer that processes the sequence
        self.rnn = nn.RNN(embed_dim, hidden_size, batch_first=True)

        # Fully connected layer to get the final classification (0 or 1)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x, lengths):
        # 1. Look up embeddings: (batch, seq_len) -> (batch, seq_len, embed_dim)
        embedded = self.embedding(x)

        # 2. Pack the sequence: ignores the padding during RNN calculation
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        # 3. RNN pass: h_n is the final hidden state we discussed in Part B
        packed_output, h_n = self.rnn(packed)

        # 4. Use the last hidden state for classification
        # h_n shape is (num_layers, batch, hidden_size)
        last_hidden = h_n[-1]

        return self.fc(last_hidden)

In [17]:
# Hyperparameters
VOCAB_SIZE = 5 # A, C, G, T, <PAD>
EMBED_DIM = 16
HIDDEN_SIZE = 64
NUM_CLASSES = 2
PAD_IDX = char2idx["<PAD>"]

model = DNAClassifier(VOCAB_SIZE, EMBED_DIM, HIDDEN_SIZE, NUM_CLASSES, PAD_IDX)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.005)

In [18]:
def get_accuracy(model, X, y, lengths):
    model.eval()
    with torch.no_grad():
        outputs = model(X, lengths)
        _, predicted = torch.max(outputs, 1)
        correct = (predicted == y).sum().item()
        return correct / y.size(0)

# Simple training loop
epochs = 200
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad()

    # Forward pass
    outputs = model(X_train, len_train)
    loss = criterion(outputs, y_train)

    # Backward pass
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        val_acc = get_accuracy(model, X_val, y_val, len_val)
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}, Val Acc: {val_acc:.4f}")

# Final test accuracy
test_acc = get_accuracy(model, X_test, y_test, len_test)
print(f"\nFinal Test Accuracy: {test_acc:.4f}")

Epoch [10/200], Loss: 0.6799, Val Acc: 0.5160
Epoch [20/200], Loss: 0.6620, Val Acc: 0.5760
Epoch [30/200], Loss: 0.6043, Val Acc: 0.6940
Epoch [40/200], Loss: 0.6500, Val Acc: 0.5720
Epoch [50/200], Loss: 0.5969, Val Acc: 0.6700
Epoch [60/200], Loss: 0.6719, Val Acc: 0.5140
Epoch [70/200], Loss: 0.6679, Val Acc: 0.4780
Epoch [80/200], Loss: 0.6533, Val Acc: 0.5000
Epoch [90/200], Loss: 0.6373, Val Acc: 0.5100
Epoch [100/200], Loss: 0.6212, Val Acc: 0.5240
Epoch [110/200], Loss: 0.5970, Val Acc: 0.5580
Epoch [120/200], Loss: 0.3584, Val Acc: 0.7760
Epoch [130/200], Loss: 0.3635, Val Acc: 0.9280
Epoch [140/200], Loss: 1.3160, Val Acc: 0.6140
Epoch [150/200], Loss: 0.7795, Val Acc: 0.4780
Epoch [160/200], Loss: 0.5618, Val Acc: 0.7080
Epoch [170/200], Loss: 0.5765, Val Acc: 0.7080
Epoch [180/200], Loss: 0.5044, Val Acc: 0.7500
Epoch [190/200], Loss: 0.4368, Val Acc: 0.7860
Epoch [200/200], Loss: 0.4076, Val Acc: 0.7820

Final Test Accuracy: 0.7960


###C4. Generalization Tests


In [19]:
def predict_dna(model, sequence, mapping):
    model.eval()
    with torch.no_grad():
        # Encode and add batch dimension
        encoded = encode_sequence(sequence, mapping).unsqueeze(0)
        length = torch.tensor([len(sequence)])

        # Get prediction
        output = model(encoded, length)
        # Apply softmax to get probabilities
        probabilities = torch.softmax(output, dim=1)
        prob_positive = probabilities[0][1].item()

        prediction = torch.argmax(output, dim=1).item()
        return prediction, prob_positive

# Test Cases
test_cases = [
    ("ATG" + "C" * 40, 1, "Beginning"),
    ("C" * 20 + "ATG" + "G" * 20, 1, "Middle"),
    ("A" * 40 + "ATG", 1, "End"),
    ("C" * 70, 0, "No motif"),
    ("AT" + "A" * 10 + "G", 0, "Broken motif")
]

print(f"{'Sequence Type':<15} | {'True':<5} | {'Pred':<5} | {'Probability'}")
print("-" * 50)

for seq, true_label, desc in test_cases:
    pred, prob = predict_dna(model, seq, char2idx)
    print(f"{desc:<15} | {true_label:<5} | {pred:<5} | {prob:.4f}")

Sequence Type   | True  | Pred  | Probability
--------------------------------------------------
Beginning       | 1     | 1     | 0.7038
Middle          | 1     | 1     | 0.7826
End             | 1     | 0     | 0.3119
No motif        | 0     | 0     | 0.0160
Broken motif    | 0     | 0     | 0.0707


###Follow-Up Questions for Understanding

1. Why do we use nn.Embedding before the RNN?
It converts discrete nucleotide indices (0-4) into continuous dense vectors. This allows the model to learn a richer representation of the DNA bases in a high-dimensional space, which is more effective for training than raw integers.

2. Why is this a many-to-one task?
The model takes a sequence of many inputs (DNA bases) and produces a single output (a classification label 0 or 1).

3. Why do we use the final hidden state for classification?
The final hidden state $h_n$ serves as the "memory summary" of the entire sequence. By the last step, it should ideally contain all the information needed to determine if the motif was present anywhere in the string.

4. Why do we pad sequences in a batch?
Neural networks require batch inputs to have a consistent shape. Since your sequences vary from 30 to 80 characters, padding with a <PAD> token (index 4) ensures every sequence in the batch has the same length.

5. Why does pack_padded_sequence help when sequences have different lengths?
It informs the RNN about the actual length of each sequence so it can stop processing at the real end. This prevents the "memory" from being corrupted by useless padding tokens and ensures the final hidden state is captured correctly.

6. What kind of information might a vanilla RNN forget on long sequences?
A vanilla RNN often forgets information from the very beginning of a long sequence because the signal decays over time. In your first test, the model struggled to detect a motif at the "Beginning" for this exact reason.

7. What problem are LSTMs and GRUs designed to reduce compared with a vanilla RNN?
They are designed to mitigate the Vanishing Gradient Problem. These architectures use special internal mechanisms to maintain a stable "memory" over much longer time steps.

8. What do gates help an LSTM or GRU decide while reading a sequence?Gates (Input, Forget, and Output) act as regulators that decide what new information to store, what irrelevant past information to discard (forget), and what parts of the memory to use for the current output.

9. If you replaced nn.RNN with nn.LSTM, what extra value does PyTorch return?In addition to the hidden state ($h_n$), an LSTM returns a Cell State ($c_n$). The cell state acts like a "long-term memory" conveyor belt that stays relatively unchanged throughout the sequence.